In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import importlib
import src.features
import src.quality_checks
import src.pipeline

importlib.reload(src.features)
importlib.reload(src.quality_checks)
importlib.reload(src.pipeline)

from src.pipeline import run_pipeline
from src.db import run_query

outputs = run_pipeline(refresh_data=False)

Step 1/7 — Creating schemas...
Step 2/7 — Reusing existing raw parquet files...
Step 3/7 — Loading raw parquet files into DuckDB...
Step 4/7 — Validating raw data project scope...
Running raw project-scope gate...
Raw project-scope checks:
- historical_required_columns: PASS
- forecast_required_columns: PASS
- historical_date_parse: PASS
- forecast_date_parse: PASS
- historical_city_scope: PASS
- forecast_city_scope: PASS
- historical_date_range: PASS
- forecast_date_range: PASS
- historical_rows_per_city: PASS
- forecast_rows_per_city: PASS
Raw project-scope gate passed.
Step 5/7 — Cleaning data, running quality checks, and building model features...
Running quality gate on cleaned historical data...
Quality checks:
- missing_values: PASS
- duplicate_rows: PASS
- duplicate_city_dates: PASS
- missing_dates: PASS
- weather_ranges: PASS
Quality gate passed.
Step 6/7 — Training models and building final 28-day forecast...
Training GradientBoosting model for ML horizon=1...
Training Gradie

In [2]:
run_query("""
SELECT table_schema, table_name
FROM information_schema.tables
ORDER BY table_schema, table_name
""")

,table_schema,table_name
0,analytics,final_28d_forecast
1,analytics,model_features
2,raw,forecast
3,raw,historical


In [3]:
run_query("""
SELECT *
FROM analytics.final_28d_forecast
ORDER BY city, target_time
LIMIT 35
""")

,city,origin_time,forecast_horizon,target_time,source,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,sunshine_duration
0,Baku,2026-05-02,1,2026-05-03,api_forecast,13.500000,0.000000,28.300000,81.000000,92.000000,7743.140000
1,Baku,2026-05-02,2,2026-05-04,api_forecast,16.400000,0.000000,24.900000,84.000000,89.000000,32576.210000
2,Baku,2026-05-02,3,2026-05-05,api_forecast,18.600000,0.000000,18.800000,78.000000,57.000000,43280.600000
3,Baku,2026-05-02,4,2026-05-06,api_forecast,18.600000,0.000000,18.200000,76.000000,50.000000,43052.130000
4,Baku,2026-05-02,5,2026-05-07,api_forecast,18.500000,0.000000,20.700000,71.000000,71.000000,40470.820000
5,Baku,2026-05-02,6,2026-05-08,api_forecast,23.100000,0.000000,21.600000,66.000000,28.000000,49348.050000
6,Baku,2026-05-02,7,2026-05-09,api_forecast,22.200000,0.000000,19.100000,66.000000,15.000000,49380.280000
7,Baku,2026-05-09,8,2026-05-10,ml_model,22.260044,0.680973,21.997531,69.694351,48.089580,40728.567862
8,Baku,2026-05-09,9,2026-05-11,ml_model,21.692092,1.348811,23.810881,70.607979,57.513256,36731.221354
9,Baku,2026-05-09,10,2026-05-12,ml_model,20.603266,3.519672,24.051091,70.807845,56.176567,37876.745255


In [4]:
run_query("""
SELECT city, source, COUNT(*) AS rows
FROM analytics.final_28d_forecast
GROUP BY city, source
ORDER BY city, source
""")

,city,source,rows
0,Baku,api_forecast,7
1,Baku,ml_model,21
2,Gabala,api_forecast,7
3,Gabala,ml_model,21
4,Guba,api_forecast,7
5,Guba,ml_model,21
6,Lankaran,api_forecast,7
7,Lankaran,ml_model,21
8,Shaki,api_forecast,7
9,Shaki,ml_model,21
